# Числове дослідження МНС і ПАРТАН-МНС

Блокнот зібрано за пунктами ТЗ. На кожному етапі виводиться таблиця результатів і графік за кількістю викликів цільової функції або траєкторією пошуку.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "optimization").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import optimization.experiments as experiments
experiments = importlib.reload(experiments)

BASE_PARAMS = experiments.BASE_PARAMS
EXPERIMENTS = experiments.EXPERIMENTS
METHODS = experiments.METHODS
DISPLAY_COLUMN_LABELS = experiments.DISPLAY_COLUMN_LABELS
compare_methods = experiments.compare_methods
compare_penalty_methods = experiments.compare_penalty_methods
old_sympy_check = experiments.old_sympy_check
sweep = experiments.sweep
from optimization.functions import F_MIN, FUNCTION_FORMULA, X_MIN, X_START, power_function
from optimization.partan_steepest_descent import partan_steepest_descent
from optimization.plots import plot_trajectory
from optimization.steepest_descent import steepest_descent

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
plt.rcParams["figure.figsize"] = (8, 4)

base_params = dict(BASE_PARAMS)
base_params

In [ ]:
COLUMN_LABELS = {
    "method": "метод",
    "parameter_value": "значення параметра",
    "x_final": "кінцева точка",
    "f_final": "кінцеве значення функції",
    "grad_norm_final": "норма градієнта",
    "iterations": "кількість ітерацій",
    "func_calls": "кількість викликів функції",
    "status": "статус",
}
COLUMN_LABELS.update(DISPLAY_COLUMN_LABELS)

STATUS_LABELS = {
    "converged": "збіжність",
    "max_iter": "ліміт ітерацій",
    "numerical_issue": "числова помилка",
}

PARAMETER_LABELS = {
    "derivative_h": "крок h",
    "gradient_scheme": "схема числового диференціювання",
    "line_search_method": "метод одновимірного пошуку",
    "line_search_eps": "точність одновимірного пошуку",
    "sven_alpha": "параметр α методу Свена",
    "stop_criterion": "критерій зупинки",
}


def show_table(df):
    display(df.rename(columns=COLUMN_LABELS))


def format_vector(x):
    arr = np.asarray(x, dtype=float).reshape(-1)
    return "[" + ", ".join(f"{value:.8g}" for value in arr) + "]"


def is_bar_plot(parameter_name: str):
    values = EXPERIMENTS[parameter_name]
    return len(values) <= 3 or any(isinstance(value, str) for value in values)


def show_sweep(parameter_name: str, title: str):
    frames = []
    for method_name, method_fn in METHODS.items():
        df = sweep(method_fn, parameter_name, EXPERIMENTS[parameter_name], base_params=base_params)
        df.insert(0, "method", method_name)
        frames.append(df)

    table = pd.concat(frames, ignore_index=True)
    display(Markdown(f"## {title}"))
    show_table(table)

    values = [str(value) for value in EXPERIMENTS[parameter_name]]
    x = np.arange(len(values))
    fig, ax = plt.subplots(figsize=(8, 4))
    grouped = list(table.groupby("method", sort=False))
    if is_bar_plot(parameter_name):
        width = 0.34
        offsets = np.linspace(-width / 2, width / 2, len(grouped))
        for offset, (method_name, group) in zip(offsets, grouped):
            ax.bar(x + offset, group["func_calls"].to_numpy(), width=width, label=method_name)
        ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    else:
        for method_name, group in grouped:
            ax.plot(x, group["func_calls"].to_numpy(), "o-", linewidth=1.8, label=method_name)
        ax.grid(True, linestyle="--", alpha=0.35)

    ax.set_xticks(x)
    ax.set_xticklabels(values, rotation=20, ha="right")
    ax.set_xlabel(PARAMETER_LABELS.get(parameter_name, parameter_name))
    ax.set_ylabel("Кількість викликів функції")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()
    return table


def show_method_comparison():
    display(Markdown("## 12.7. Порівняння МНС і ПАРТАН-МНС"))
    table = compare_methods(base_params)
    show_table(table)

    x = np.arange(len(table))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x, table["func_calls"].to_numpy(), color=["tab:blue", "tab:orange"])
    ax.set_xticks(x)
    ax.set_xticklabels(table["method"].to_list(), rotation=15, ha="right")
    ax.set_ylabel("Кількість викликів функції")
    ax.set_title("Порівняння методів за кількістю викликів функції")
    ax.grid(True, axis="y", linestyle="--", alpha=0.35)
    plt.tight_layout()
    plt.show()
    return table


def show_trajectories():
    display(Markdown("## 14. Траєкторії пошуку"))
    mns_result = steepest_descent(power_function, X_START, **base_params)
    partan_result = partan_steepest_descent(power_function, X_START, **base_params)

    table = pd.DataFrame([
        {
            "method": "МНС",
            "x_final": format_vector(mns_result["x_final"]),
            "f_final": mns_result["f_final"],
            "grad_norm_final": mns_result["grad_norm_final"],
            "iterations": mns_result["iterations"],
            "func_calls": mns_result["func_calls"],
            "status": STATUS_LABELS.get(mns_result["status"], mns_result["status"]),
        },
        {
            "method": "ПАРТАН-МНС",
            "x_final": format_vector(partan_result["x_final"]),
            "f_final": partan_result["f_final"],
            "grad_norm_final": partan_result["grad_norm_final"],
            "iterations": partan_result["iterations"],
            "func_calls": partan_result["func_calls"],
            "status": STATUS_LABELS.get(partan_result["status"], partan_result["status"]),
        },
    ])
    show_table(table)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    plot_trajectory(power_function, mns_result["points"], "Траєкторія МНС", cmap="Blues", ax=axes[0])
    plot_trajectory(power_function, partan_result["points"], "Траєкторія ПАРТАН-МНС", cmap="Oranges", ax=axes[1])
    plt.tight_layout()
    plt.show()
    return table, mns_result, partan_result

## 2. Початкова функція та стартові дані

In [ ]:
initial_table = pd.DataFrame([
    {"назва": "функція", "значення": FUNCTION_FORMULA},
    {"назва": "x_start", "значення": format_vector(X_START)},
    {"назва": "x_min", "значення": format_vector(X_MIN)},
    {"назва": "f_min", "значення": F_MIN},
    {"назва": "f(x_start)", "значення": power_function(X_START)},
])
display(initial_table)

x1 = np.linspace(-1.5, 1.5, 260)
x2 = np.linspace(-0.5, 1.5, 260)
X, Y = np.meshgrid(x1, x2)
Z = np.vectorize(lambda a, b: power_function(np.array([a, b])))(X, Y)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(X, Y, Z, levels=30, cmap="viridis")
ax.scatter([X_START[0]], [X_START[1]], color="tab:red", label="x_start")
ax.scatter([X_MIN[0]], [X_MIN[1]], color="tab:green", label="x_min")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("Лінії рівня цільової функції")
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
h_table = show_sweep("derivative_h", "12.1. Вплив кроку h числового диференціювання")

In [ ]:
scheme_table = show_sweep("gradient_scheme", "12.2. Вплив схеми числового диференціювання")

In [ ]:
line_search_method_table = show_sweep("line_search_method", "12.3. Вплив методу одновимірного пошуку")

In [ ]:
line_search_eps_table = show_sweep("line_search_eps", "12.4. Вплив точності одновимірного пошуку")

In [ ]:
sven_alpha_table = show_sweep("sven_alpha", "12.5. Вплив параметра методу Свена")

In [ ]:
stop_criterion_table = show_sweep("stop_criterion", "12.6. Вплив критерію зупинки")

In [ ]:
comparison_table = show_method_comparison()

In [ ]:
trajectory_table, mns_result, partan_result = show_trajectories()

## Метод штрафних функцій для умовної оптимізації

Для умовної оптимізації використовується метод штрафних функцій, а саме метод зовнішньої точки. За основу взято степеневу функцію, яка вже використовувалась у задачі безумовної оптимізації.

Допустима область задається обмеженням:

$$x_1^2 + x_2^2 \le 1$$

Ця область є випуклою. Безумовний мінімум функції знаходиться в точці $$(1;1)$$, яка не належить допустимій області, оскільки:

$$1^2 + 1^2 = 2 > 1$$

Тому задача відповідає випадку розташування локального мінімуму поза випуклою допустимою областю.

Штрафна функція має вигляд:

$$F(x,r)=f(x)+r\max(0, x_1^2+x_2^2-1)^2$$


In [ ]:
penalty_params = dict(base_params)
penalty_params["stop_criterion"] = "combined"
penalty_tables = compare_penalty_methods(base_params=penalty_params)


In [ ]:
display(Markdown("### МНС"))
show_table(penalty_tables["МНС"])

display(Markdown("### ПАРТАН-МНС"))
show_table(penalty_tables["ПАРТАН-МНС"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for method_name, table in penalty_tables.items():
    axes[0].plot(table["r"], table["violation"], "o-", linewidth=1.8, label=method_name)
    axes[1].plot(table["r"], table["func_calls"], "o-", linewidth=1.8, label=method_name)

axes[0].set_xscale("log")
axes[0].set_yscale("symlog", linthresh=1e-12)
axes[0].set_xlabel("r")
axes[0].set_ylabel("Порушення обмеження")
axes[0].set_title("Зміна порушення залежно від r")
axes[0].grid(True, which="both", linestyle="--", alpha=0.35)

axes[1].set_xscale("log")
axes[1].set_xlabel("r")
axes[1].set_ylabel("Кількість викликів функції")
axes[1].set_title("Виклики функції на етапах штрафу")
axes[1].grid(True, which="both", linestyle="--", alpha=0.35)

for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
penalty_summary = pd.DataFrame([
    {
        "method": method_name,
        "r": table.iloc[-1]["r"],
        "x_final": table.iloc[-1]["x_final"],
        "f_original": table.iloc[-1]["f_original"],
        "constraint_value": table.iloc[-1]["constraint_value"],
        "violation": table.iloc[-1]["violation"],
        "func_calls": int(table["func_calls"].sum()),
    }
    for method_name, table in penalty_tables.items()
])
show_table(penalty_summary)

best_method = penalty_summary.loc[penalty_summary["func_calls"].idxmin(), "method"]
display(Markdown(
    f"За сумарною кількістю викликів функції менше обчислень дав метод **{best_method}**. "
    "Для коректної роботи штрафного методу потрібно дивитися на `constraint_value <= 0` "
    "та зменшення `violation` при зростанні коефіцієнта `r`."
))


## 14.1. Контроль через старі SymPy-версії методів

Цей блок потрібен лише для перевірки траєкторій. Старі реалізації використовують аналітичні похідні та точний пошук кроку через SymPy, тому кількість ітерацій обмежена.

In [ ]:

old_check = old_sympy_check(max_iter_mns=10, max_iter_partan=10, eps=1e-6)
current_check = compare_methods(BASE_PARAMS)
current_check["джерело"] = "поточна NumPy-версія"
old_table = old_check["table"].copy()
old_table["джерело"] = "стара SymPy-версія"
show_table(pd.concat([current_check, old_table], ignore_index=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_trajectory(power_function, old_check["mns_points"], "Контрольна траєкторія МНС (SymPy)", cmap="Greens", ax=axes[0])
plot_trajectory(power_function, old_check["partan_points"], "Контрольна траєкторія ПАРТАН-МНС (SymPy)", cmap="Purples", ax=axes[1])
plt.tight_layout()
plt.show()